# 00 · Setup & Sanity Checks
### Amazon Bedrock AgentCore — Zero → Hero (hands-on series)

You already know **Bedrock Agents**, the console **Agent Builder**, and the **Strands SDK**. This series adds the missing layer: **AgentCore** — the production substrate that runs *your* agent code (any framework, any model) without you provisioning ECS/EKS, load balancers, IAM plumbing, or CloudWatch by hand.

**Mental model (one line):** Direct model calls → Bedrock Agents (AWS runs the loop) → **AgentCore (you bring the code, AWS runs the infra)**.

```mermaid
flowchart LR
    A[Layer 1: bedrock-runtime<br/>Converse / InvokeModel<br/>you write the whole loop] --> B[Layer 2: Bedrock Agents / Agent Builder<br/>config-first, AWS runs the ReAct loop]
    B --> C[Layer 3: AgentCore<br/>bring your own code + framework<br/>AWS runs production infra]
```

This notebook gets your environment provably working **before** you touch any agent code. Run every cell top to bottom. If a check fails, the fix is right next to it.

> **Region:** `us-east-1` for the entire series. **Models:** Claude **Haiku 4.5** (cheap/fast) and **Sonnet 4.5** (more reasoning). No ambiguity, no guessing.


## 1. What you need (once)

- An **AWS account** with credentials you can use locally (`aws configure`) or paste into Colab.
- **Model access** enabled for Anthropic Claude in `us-east-1` (Bedrock console → *Model access* → enable Claude 4.x). Without this, every call returns `AccessDeniedException`.
- For the Runtime notebook (02) you'll also want the managed policies **`BedrockAgentCoreFullAccess`** + **`AmazonBedrockFullAccess`** on your principal, and **Node.js 20+** if you use the AgentCore CLI. We'll flag that when we get there.

Nothing else is required to start. Memory, Gateway, Identity, etc. are provisioned on demand in their own notebooks.


## 2. Install the stack

Pick the block for your environment. Both install the same packages.

**What each package is:**
- `strands-agents` — the agent framework (the `Agent`, `@tool`, multi-agent primitives).
- `strands-agents-tools` — ready-made tools incl. AgentCore **Code Interpreter** and **Browser** wrappers.
- `bedrock-agentcore` — the AgentCore **runtime SDK** (`BedrockAgentCoreApp`) + **Memory** client.
- `bedrock-agentcore-starter-toolkit` — the Python helper that deploys an agent to Runtime *from a notebook* (`Runtime().configure/launch/invoke`).
- `boto3` — raw AWS SDK; we use it for the "under the hood" calls (`bedrock-runtime`, `bedrock-agentcore`, control planes).


In [ ]:
# === VS Code / local (run in a terminal, or uncomment to run here) ===
# %python -m venv .venv && source .venv/bin/activate     # Windows: .venv\Scripts\activate
%pip install -U strands-agents strands-agents-tools bedrock-agentcore bedrock-agentcore-starter-toolkit boto3
!python -m ipykernel install --user --name agentcore --display-name "Python (agentcore)"
#   -> then in VS Code: select the "Python (agentcore)" kernel (top-right) for this notebook.

# === Google Colab (uncomment) ===
# !pip install -q -U strands-agents strands-agents-tools bedrock-agentcore bedrock-agentcore-starter-toolkit boto3

# Verify versions (these are the versions this series is written and tested against):
import importlib.metadata as m
for pkg in ["strands-agents", "strands-agents-tools", "bedrock-agentcore",
            "bedrock-agentcore-starter-toolkit", "boto3"]:
    try:
        print(f"{pkg:38s} {m.version(pkg)}")
    except m.PackageNotFoundError:
        print(f"{pkg:38s} NOT INSTALLED  <-- run the install block above")

Note: you may need to restart the kernel to use updated packages.


UsageError: Line magic function `%%python` not found.


## 3. Credentials & region

AgentCore is just AWS — it uses your normal credential chain. **Never hardcode keys in a notebook you'll share.**

- **VS Code / local:** run `aws configure` once (writes `~/.aws/credentials`), or export `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` / `AWS_SESSION_TOKEN` in your shell. boto3 finds them automatically.
- **Colab:** use the 🔑 *Secrets* panel (left sidebar) to store the keys, then load them as shown.

The cell below does **not** print your secrets — it only confirms boto3 can see *a* region.


In [ ]:
import os

# --- Colab only: pull creds from the Secrets panel (uncomment) ---
# from google.colab import userdata
# os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
# os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
# # only if using temporary creds:
# # os.environ["AWS_SESSION_TOKEN"]   = userdata.get("AWS_SESSION_TOKEN")

# One region for the whole series.
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
REGION = "us-east-1"
print("Region set to:", REGION)

## 4. The model IDs (and why they look weird)

```python
MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_SONNET = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
```

The leading **`us.`** is **not** a typo. It's a **cross-region inference profile** prefix. Claude 4.x models on Bedrock are served through inference profiles that route your request across US regions for capacity. If you pass the bare `anthropic.claude-...` id, you'll get a `ValidationException` telling you an inference profile is required. Always use the `us.` form in `us-east-1`.

We use a single config block so every later notebook references the same constants.


In [ ]:
# Cross-region inference profile IDs (required for Claude 4.x in us-east-1)
MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # cheap, fast -> default for teaching
MODEL_SONNET = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"  # stronger reasoning -> orchestration

# Default model for most cells in this series
MODEL = MODEL_HAIKU
print("Using model:", MODEL)

## 5. Sanity check A — *who am I?*

`get_caller_identity` is the cheapest possible AWS call. If this fails, it's a credentials problem, not an AgentCore problem — fix it before going further.


In [ ]:
import boto3
from botocore.exceptions import ClientError, NoCredentialsError

try:
    sts = boto3.client("sts", region_name=REGION)
    ident = sts.get_caller_identity()
    print("Account :", ident["Account"])
    print("ARN     :", ident["Arn"])
except NoCredentialsError:
    print("FAIL: boto3 found no credentials. Run `aws configure` or set env vars (cell 3).")
except ClientError as e:
    print("FAIL:", e.response["Error"]["Code"], "-", e.response["Error"]["Message"])

## 6. Sanity check B — is Claude actually reachable?

Two things can be wrong even with valid credentials:
1. **Model access not enabled** → `AccessDeniedException`.
2. **Wrong/legacy model id** → `ValidationException`.

We test the *definitive* signal: a tiny **Converse** call (Layer 1 from the mental model). This is the throwback to "direct model calls" — no agent, no framework, just `bedrock-runtime`. If this returns text, your account can run the whole series.

**The Converse request shape**, decoded:
- `modelId` — the `us.` inference profile id.
- `messages` — a list; each message has a `role` and a `content` list of blocks (`{"text": ...}`).
- `inferenceConfig` — generation knobs. For Claude 4.x, set **temperature *or* top_p, never both** (the model rejects both together). Here we only set `maxTokens`.
- Response text lives at `output → message → content[0] → text`.


In [ ]:
def converse_ping(model_id: str, region: str = REGION) -> str:
    # Smallest possible real call. Returns the model's text, or raises with a clear reason.
    rt = boto3.client("bedrock-runtime", region_name=region)
    resp = rt.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": "Reply with exactly: pong"}]}],
        inferenceConfig={"maxTokens": 10},  # note: no temperature AND top_p together for Claude 4.x
    )
    return resp["output"]["message"]["content"][0]["text"].strip()

try:
    print("Haiku  ->", converse_ping(MODEL_HAIKU))
    print("Sonnet ->", converse_ping(MODEL_SONNET))
    print("\nGreen across the board. You're ready.")
except ClientError as e:
    code_ = e.response["Error"]["Code"]
    msg = e.response["Error"]["Message"]
    print("FAIL:", code_, "-", msg)
    if code_ == "AccessDeniedException":
        print("FIX: Bedrock console (us-east-1) -> Model access -> enable Anthropic Claude.")
    elif code_ == "ValidationException":
        print("FIX: keep the 'us.' inference-profile prefix; check the model id spelling.")

## 7. Preflight guard (use this before any live class / demo)

Throttling and *legacy* model lifecycle states are the two things that bite during a live session with many users. This guard does two cheap things:

1. **Lifecycle scan** — lists Anthropic foundation models and flags any whose lifecycle is `LEGACY` (a model that still answers today but is on the way out).
2. **Live ping** — the Converse call from above (the real proof).

Run it at the top of class. If it's green, the room is green.


In [ ]:
def preflight(region: str = REGION, models=(MODEL_HAIKU, MODEL_SONNET)) -> bool:
    ok = True
    # 1) lifecycle scan (control-plane 'bedrock' client, NOT bedrock-runtime)
    b = boto3.client("bedrock", region_name=region)
    try:
        fms = b.list_foundation_models(byProvider="Anthropic")["modelSummaries"]
        legacy = [f["modelId"] for f in fms
                  if f.get("modelLifecycle", {}).get("status") == "LEGACY"]
        if legacy:
            print(f"NOTE: {len(legacy)} Anthropic model(s) marked LEGACY (still work, plan to migrate).")
    except ClientError as e:
        print("lifecycle scan skipped:", e.response["Error"]["Code"])

    # 2) live ping per model (the decisive check)
    for mid in models:
        try:
            converse_ping(mid, region)
            print(f"OK   {mid}")
        except ClientError as e:
            ok = False
            print(f"DOWN {mid} -> {e.response['Error']['Code']}")
    return ok

print("\nPREFLIGHT:", "PASS ✅" if preflight() else "FAIL ❌ — fix before proceeding")

## 8. Common setup errors → fixes

| Symptom | Cause | Fix |
|---|---|---|
| `NoCredentialsError` | boto3 sees no creds | `aws configure`, or set env vars / Colab secrets (cell 3) |
| `AccessDeniedException` on Converse | Model access not enabled | Bedrock console → *Model access* → enable Claude 4.x in `us-east-1` |
| `ValidationException: ... inference profile` | Used bare model id | Keep the **`us.`** prefix |
| `ValidationException: temperature and top_p` | Set both knobs | Set only one for Claude 4.x |
| `ThrottlingException` under class load | Per-account TPS limits | Back off + retry; request a quota increase before the session |
| Region mismatch / model "not found" | Client in wrong region | Pass `region_name="us-east-1"` everywhere (we do) |

> **Production note (carry this through the series):** in production you don't paste keys — you attach an **IAM role** to the compute (Lambda, ECS, or the AgentCore execution role), never hardcode region/secrets, scope policies to **least privilege**, and wrap AWS calls with **retries + observability**. The notebooks use explicit clients so the wiring is visible; production hides it behind roles.


## You're set. Next:

**`01_build_an_agent_three_ways.ipynb`** — build the *same* tiny agent three ways and feel the difference:
1. **No framework** — a raw Converse tool-use loop (you write everything).
2. **Bedrock Agents / Agent Builder** — config-first, AWS runs the loop.
3. **Strands** — code-first, a few lines.

Same task, three philosophies. That contrast is the whole point of AgentCore.
